In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [2]:
# Install required libraries for Qwen3-VL-2B-Instruct
!pip install -q torchvision accelerate transformers>=4.45.0

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

QWEN_MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"

# Initialize processor and model
processor_qwen = AutoProcessor.from_pretrained(QWEN_MODEL_NAME)

model_qwen = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_qwen.eval()

print("Qwen3-VL-2B-Instruct loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.26GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3-VL-2B-Instruct loaded successfully!


In [3]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_path"] == target_name:
            return record["question"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [4]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [5]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [6]:

def evaluate(query_text, b64_str):
    # Decode the base64 string to a PIL Image
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_qwen(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model_qwen.device)

    with torch.inference_mode():
        output_ids = model_qwen.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_qwen.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [8]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm

IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/annotations.json"

# Load annotations
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Filter out records where the query is "Is there a person in the image?"
filtered_annotations = [
    record for record in annotations_list
    if record.get("question", "").strip() != "Is there a person in the image?"
]

# Create a lookup map for ground truth from the filtered records
ground_truth_map = {}
for record in filtered_annotations:
    if record.get("image_path", "").startswith("images/"):
        img_id = record["image_id"]
        ground_truth_map[int(img_id)] = record["label"]

# Get list of all images in the directory
all_image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

# We only want to process images that actually exist in our filtered ground truth map
image_files = []
for filename in all_image_files:
    try:
        image_id = int(os.path.splitext(filename)[0])
        if image_id in ground_truth_map:
            image_files.append(filename)
    except ValueError:
        continue

print(f"Total raw annotations: {len(annotations_list)}")
print(f"Filtered annotations (excluding 'person' queries): {len(filtered_annotations)}")
print(f"Final count of rows to evaluate (matching image files): {len(image_files)}")

results = []
print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under original condition
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val
    })

df_results = pd.DataFrame(results)
display(df_results.head(10))


Total raw annotations: 491
Filtered annotations (excluding 'person' queries): 315
Final count of rows to evaluate (matching image files): 315
Processing 315 images through all variations...


  0%|          | 0/315 [00:00<?, ?it/s]

,Image_Id,Ground_truth,Original
0,102439,yes,Yes
1,108189,yes,Yes
2,579231,yes,Yes
3,75591,yes,Yes
4,514248,yes,Yes
5,516601,yes,No
6,526321,yes,Yes
7,207205,yes,Yes
8,280734,yes,Yes
9,468997,yes,Yes


In [10]:
df_results.rename(columns={"Original": "Qwen-2B"}, inplace=True)
df_results

,Image_Id,Ground_truth,Qwen-2B
0,102439,yes,Yes
1,108189,yes,Yes
2,579231,yes,Yes
3,75591,yes,Yes
4,514248,yes,Yes
...,...,...,...
310,17379,yes,Yes
311,399702,yes,Yes
312,418680,yes,Yes
313,172648,yes,Yes


In [11]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/diff-models-evals-on-pope/diff_models_eval (1).csv", usecols=["Image_Id","Ground_truth","Qwen-2B"])
df

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
495,574356,No,No
496,574837,No,No
497,575121,No,No
498,575916,No,No


In [12]:
df2 = pd.concat([df,df_results], ignore_index=True)
df2

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
810,17379,yes,Yes
811,399702,yes,Yes
812,418680,yes,Yes
813,172648,yes,Yes


In [13]:
df2 = df2.drop_duplicates(subset=["Image_Id"], keep="first")
df2

,Image_Id,Ground_truth,Qwen-2B
0,6578,No,No
1,8602,No,No
2,9469,No,No
3,9712,No,Yes
4,10733,No,No
...,...,...,...
810,17379,yes,Yes
811,399702,yes,Yes
812,418680,yes,Yes
813,172648,yes,Yes


In [14]:
df2.to_csv("qwen-2B-without-person.csv")